In [ ]:
#Load required libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer
import seaborn as sns
import os
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load the dataset
landmarks_and_places_df = pd.read_csv('landmarks-and-places-of-interest-including-schools-theatres-health-services-spor.csv')
art_places_df = pd.read_csv('public-artworks-fountains-and-monuments.csv')
#Check the dataset structure
print(landmarks_and_places_df.info())
print(landmarks_and_places_df.head())
print(art_places_df.info())
print(art_places_df.head())

In [ ]:
# Check the unique theme and subtheme of the landmarks dataset
print(landmarks_and_places_df['Theme'].unique())
print(landmarks_and_places_df['Sub Theme'].unique())

# Check the duplicate entries in the landmarks dataset
print(landmarks_and_places_df.value_counts(subset=['Feature Name']) > 1)

# remove duplicates
landmarks_and_places_df1 = landmarks_and_places_df.drop_duplicates(subset=['Feature Name'])
print(landmarks_and_places_df1.value_counts(subset=['Feature Name']) > 1)


In [ ]:
#Check the unique categories of the art places dataset
print(art_places_df['Asset Type'].unique())

#Check the repeated name in the art places dataset  
print(art_places_df['Name'].value_counts())

#Drop duplicates in the art places dataset based on 'Name' and 'Mel way Ref' columns
art_places_df1 = art_places_df.drop_duplicates(subset=['Name', 'Mel way Ref'])
print(art_places_df1.info())

In [ ]:
# Check the distribution of coordinates in two datasets, visualize them on a scatter plot to identify any outliers or clustering patterns.
# Melbourne bounding box
MELB_LON_MIN, MELB_LON_MAX = 144.5, 145.5
MELB_LAT_MIN, MELB_LAT_MAX = -38.5, -37.5


# Use coordinates parsing function to extract lat/lon from the "Co-ordinates" column
def parse_coordinates(df):
    d = df.copy()
    d.columns = [c.strip() for c in d.columns]

    parts = d["Co-ordinates"].astype(str).str.split(",", n=1, expand=True)

    lat = pd.to_numeric(parts[0].str.strip(), errors="coerce")
    lon = pd.to_numeric(parts[1].str.strip(), errors="coerce")

    return lat, lon


# Validation function to check if lat/lon are within Melbourne bounds
def get_valid_mask(lat, lon):
    melb_ok = lat.between(MELB_LAT_MIN, MELB_LAT_MAX) & lon.between(MELB_LON_MIN, MELB_LON_MAX)
    return melb_ok


# Parse coordinates and get valid masks for both datasets
lm_lat, lm_lon = parse_coordinates(landmarks_and_places_df1)
aw_lat, aw_lon = parse_coordinates(art_places_df1)

lm_valid = get_valid_mask(lm_lat, lm_lon)
aw_valid = get_valid_mask(aw_lat, aw_lon)


# Visualization
plt.figure(figsize=(8, 8))

plt.scatter(lm_lon[lm_valid], lm_lat[lm_valid], s=10, alpha=0.5, label="Landmarks")
plt.scatter(aw_lon[aw_valid], aw_lat[aw_valid], s=10, alpha=0.5, label="Artworks")

plt.xlim(MELB_LON_MIN, MELB_LON_MAX)
plt.ylim(MELB_LAT_MIN, MELB_LAT_MAX)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Melbourne Points Distribution")
plt.legend()
plt.grid(alpha=0.2)

plt.show()


# Print the number of valid coordinates in each dataset
print("Landmarks valid:", lm_valid.sum())
print("Artworks valid:", aw_valid.sum())

